In [ ]:
from finsight.ingestion.loaders import load_manifest, load_filing

manifest = load_manifest()
row = next(r for r in manifest if r["doc_name"] == "3M_2018_10K")
pages = load_filing(row)

print("Pages with text:", len(pages))
print(pages[50].metadata)
print(pages[50].page_content[:1500])

In [ ]:
import random
from finsight.ingestion.chunkers import split_documents

chunks = split_documents(pages)
print("Chunks:", len(chunks))

for c in random.sample(chunks, 3):
    print(c.metadata)
    print(c.page_content)
    print("=" * 80)

In [ ]:
hits = [c for c in chunks if "Purchases of property, plant and equipment" in c.page_content]
print(len(hits), "chunks mention capex")
print(hits[0].metadata)
print(hits[0].page_content)

In [ ]:
from finsight.indexing.store import get_vector_store

store = get_vector_store("chunks_baseline")

question = "What was 3M's capital expenditure in FY2018?"
results = store.similarity_search_with_score(question, k=5)

for doc, score in results:
    m = doc.metadata
    print(f"{score:.3f} | {m['company']} {m['fiscal_year']} p.{m['page'] + 1}")
    print("   ", doc.page_content[:200].replace("\n", " "))